# Milan mobile traffic forecasting - full pipeline (Kaggle / Colab)

This notebook runs the whole project on the real Telecom Italia data, in the same order as `run_all.sh`:
ingestion, EDA, time-series analysis, baselines, tuning rounds, final test runs, comparison and the report tables.
I wrote it because my laptop cannot hold the 20.8 GB of raw files comfortably, and Kaggle gives ~30 GB of RAM plus a free GPU for the LSTM rounds.

On Kaggle, add a dataset that contains the 62 `sms-call-internet-mi-YYYY-MM-DD.txt` files
(Add data > search "Telecom Italia" / "sms-call-internet-mi"). The next cell searches `/kaggle/input`
for those filenames, so the folder slug does not have to match anything in particular.
Check you have all 62 days; several public copies only cover one week.
Turn on a GPU under `Settings > Accelerator` if you want the LSTM rounds to go faster.

On Colab, download the files once from https://doi.org/10.7910/DVN/EGZHFV (you have to fill in the guestbook) into Google Drive,
mount Drive and point `RAW_DIR` at that folder.

In [ ]:
import os, sys, subprocess, pathlib

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

def find_raw_dir(root):
    """Return the folder that actually contains sms-call-internet-mi-*.txt files."""
    root = pathlib.Path(root)
    hits = sorted(root.rglob("sms-call-internet-mi-*.txt"))
    if not hits:
        print("nothing matching sms-call-internet-mi-YYYY-MM-DD.txt under", root)
        if root.exists():
            print("what is there:")
            for p in sorted(root.rglob("*"))[:40]:
                print(" ", p)
        return str(root)
    raw_dir = str(hits[0].parent)
    print(f"found {len(hits)} daily files in {raw_dir}")
    return raw_dir

if ON_KAGGLE:
    # don't assume the dataset slug; search whatever you attached under /kaggle/input
    RAW_DIR = find_raw_dir("/kaggle/input")
    WORK = "/kaggle/working"
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW_DIR = find_raw_dir("/content/drive/MyDrive/milan_raw")
    WORK = "/content"
else:
    RAW_DIR = find_raw_dir("data/raw")
    WORK = "."
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local", "| raw dir:", RAW_DIR)

In [ ]:
%cd {WORK}
if not pathlib.Path("Time-Series-Forecasting").exists():
    !git clone -q https://github.com/Samkwizera/Time-Series-Forecasting.git
%cd Time-Series-Forecasting
!pip install -q -r requirements.txt
!pip install -q -e .

The default config expects the raw files in `data/raw`, which is not where Kaggle puts them. I copy the config, swap in `RAW_DIR`, and leave every other path relative so the Parquet intermediates land in the writable working directory. The next cell counts the 62 daily files locally. It does not call Harvard Dataverse; that API often returns 403 from Kaggle.

In [ ]:
import yaml
cfg = yaml.safe_load(open("config/default.yaml"))
cfg["paths"]["raw_dir"] = RAW_DIR
yaml.safe_dump(cfg, open("config/kaggle.yaml", "w"), sort_keys=False)
os.environ["MILAN_CONFIG"] = "config/kaggle.yaml"

def run(script, *args):
    """Run a pipeline script and stop the notebook if it fails."""
    cmd = [sys.executable, script, "--config", "config/kaggle.yaml", *args]
    print(">", " ".join(cmd))
    subprocess.run(cmd, check=True)

run("scripts/00_download.py", "--verify")

## Stage 1 - ingestion

This is the slow part and the later cells cannot run without it. On the full 20.8 GB it often takes around an hour. Wait until the cell prints the `hourly_*.parquet` paths. If you skip it, or stop it early, EDA will look for `data/processed/citywide_10min.parquet` and fail.

In [ ]:
run("scripts/01_ingest.py")
from pathlib import Path
import pandas as pd
city = Path("data/processed/citywide_10min.parquet")
assert city.exists(), (
    f"ingest did not write {city}. Scroll up: either the raw files were missing "
    "or the ingest cell was interrupted. Re-run this cell and wait for it to finish."
)
pd.read_csv("reports/tables/memory_log.csv").tail(8)

## Stages 2-3 - exploratory and time-series analysis

The EDA script produces the citywide plots, the spatial maps, and the k-means clustering of daily profiles that picks the cells we forecast (`selected_cells.json`). The TSA script then runs the stationarity tests, STL/MSTL decomposition and ACF/PACF on those cells. I only display a handful of the figures here; the rest are in `reports/figures/`.

In [ ]:
run("scripts/02_eda.py")
run("scripts/03_tsa.py")
from IPython.display import Image, display
for f in ["eda_citywide_hourly", "eda_daily_profiles", "eda_spatial_internet", "eda_clusters_internet", "tsa_acf_citywide"]:
    display(Image(f"reports/figures/{f}.png", width=900))

## Stages 4-5 - baselines and tuning rounds

The three naive baselines come first, on both splits, so every later number has something to be compared against.

Each tuning round lives in `experiments/tuning_plan.yaml` with a `why` field that I wrote before running it, based on what the previous round (or the ACF) suggested. Every run appends a row to `experiments/experiment_log.md`, so the log reads as the actual trail of experiments rather than a summary written afterwards. SARIMA and LSTM rounds are manual; for LightGBM the manual rounds ablate the feature groups and then 20 Optuna trials handle the capacity parameters, where I did not have a strong prior. If a result suggests a new hypothesis, the loop continues by adding a round to the YAML and rerunning `05_tune.py --rounds <id>`.

In [ ]:
run("scripts/04_train.py", "--model", "baselines", "--part", "val")
run("scripts/04_train.py", "--model", "baselines", "--part", "test")
run("scripts/05_tune.py", "--model", "sarima")
run("scripts/05_tune.py", "--model", "lightgbm", "--optuna", "20")
run("scripts/05_tune.py", "--model", "lstm")
print(open("experiments/experiment_log.md").read())

## Stage 6 - final test runs and comparison

For each model I take the configuration with the lowest validation MASE and run it once on the test split (22 Dec to 1 Jan, deliberately covering the holidays). The test period is never touched before this point. `06_compare.py` then scores all models against the baselines, runs the Diebold-Mariano tests and writes the failure-analysis figures.

In [ ]:
# same selection rule as run_all.sh, written out here so the chosen runs are visible in the output
import json, glob
def best(model):
    runs = [json.load(open(p)) for p in glob.glob(f"experiments/runs/{model}/*_val.json")]
    r = min(runs, key=lambda r: r["metrics"]["mase"])
    return r["run_id"].removesuffix("_val"), r["params"]
runs = {m: best(m) for m in ["sarima", "lightgbm", "lstm"]}
for m, (rid, params) in runs.items():
    print(m, rid, {k: v for k, v in params.items() if k in ("order","seasonal_order","fourier_k","lags","num_leaves","learning_rate","cell","hidden_size","num_layers","input_window")})
for m, (rid, params) in runs.items():
    cell = params.get("cell", m) if m == "lstm" else m
    run("scripts/04_train.py", "--model", cell, "--part", "test", "--run", rid,
        "--params", json.dumps(params), "--note", "final test run of best validation config")
lstm_cell = runs["lstm"][1].get("cell", "lstm")
run("scripts/06_compare.py", "--runs", f"sarima={runs['sarima'][0]}",
    f"lightgbm={runs['lightgbm'][0]}", f"{lstm_cell}={runs['lstm'][0]}")
pd.read_csv("reports/tables/results_summary.csv")

In [ ]:
for f in ["results_mase_by_horizon_and_cell", "results_forecasts_h1", "failure_daily_error", "failure_hour_daytype"]:
    display(Image(f"reports/figures/{f}.png", width=900))

## Stage 7 - report assets

The report never hard-codes a number: `07_report_assets.py` turns the CSVs into LaTeX tables and `\newcommand` macros in `report/generated/`. The last cell zips `reports/`, `experiments/` and those generated files so I can unpack them in my local clone and build the PDF there, since Kaggle has no LaTeX.

In [ ]:
run("scripts/07_report_assets.py")
!zip -qr results_bundle.zip reports experiments report/generated
print("download results_bundle.zip, unpack it in the local clone, then run report/build.sh")